# Beat Tracking and Chord Recognition

The goal of this homework is to implement classic approaches for beat tracking and chord recognition. Through this assignment, you are expected to develop a solid understanding of onset and chroma audio features, as well as fundamental sequence modeling algorithms such as Dynamic Programming and Hidden Markov Models (HMMs). In addition, you will improve the accuracy of the basic configuration with your ideas.

## Load a music track with its beat and chord annotation

In [ ]:
# %%capture prevents the output log from displaying in this cell. If you want to see the log, remove %%capture below.
%%capture

!pip install libfmp
!pip install mir_eval

import librosa
import requests
import io
import IPython.display as ipd
import numpy as np

from matplotlib import pyplot as plt

import libfmp.b
import libfmp.c2
import libfmp.c3
import libfmp.c4
import libfmp.c5
import libfmp.c6
import mir_eval

import pandas as pd


Let's first load Beatles' Let It Be and listen to it.

In [ ]:

audio_url = "https://www.audiolabs-erlangen.de/resources/MIR/FMP/data/C5/FMP_C5_Audio_Beatles_LetItBe_Beatles_1970-LetItBe-06.wav"
response = requests.get(audio_url)
audio_bytes = io.BytesIO(response.content)

y, sr = librosa.load(audio_bytes)

# take the first 30 seconds
y_slice = y[:30*sr]

ipd.Audio(y_slice, rate=sr)

Let's visualize the beat and chord labels. Also, let's listen to the beats as a series of click sounds.

In [ ]:
# get beat labels
beat_label_url = "https://isophonics.net/files/annotations/beat/The%20Beatles/12_-_Let_It_Be/06_-_Let_It_Be.txt"
response = requests.get(beat_label_url)
df = pd.read_csv(io.BytesIO(response.content), sep='\s+', header=None, comment='[', usecols=[0])
gt_beat_label = df.values.flatten()

# get chord labels
chord_label_url = "https://www.audiolabs-erlangen.de/resources/MIR/FMP/data/C5/FMP_C5_Audio_Beatles_LetItBe_Beatles_1970-LetItBe-06_Chords_simplified.csv"
response = requests.get(chord_label_url)
gt_chord_label, _ = libfmp.c4.read_structure_annotation(io.BytesIO(response.content))

# plot chord annotations
color_ann = {'N': 'white',
             'C': 'red', 'C#': 'peru', 'D': 'orange', 'D#': 'yellow', 'Eb': 'yellow',
             'E': 'springgreen', 'F': 'cyan', 'F#': 'coral', 'G': 'blue',
             'G#': 'olive', 'A': 'teal', 'A#': 'indigo', 'Bb': 'indigo', 'B': 'pink',
             'C#:min': 'steelblue', 'C#m': 'steelblue', 'A:min': 'greenyellow', 'A:m': 'greenyellow',
             'G:min': 'olive', 'E:min': 'lightcoral', 'B:min': 'saddlebrown'}

# visualize chords blocks
fig, ax = libfmp.b.plot_segments(gt_chord_label[:12], figsize=(15, 2), time_label='Time (seconds)',
                       fontsize=15, colors=color_ann, alpha=0.5)

# plot beat positions
ax.vlines(gt_beat_label, ymin=0, ymax=1, color='Green', linestyle='-', linewidth=2, alpha=1, label='Beats')

plt.title('Chord Annotations with Beat Positions (Let It Be)')
plt.show()

# sonify the estimated beat
gt_beat_label_slice = gt_beat_label[(gt_beat_label >= 0) & (gt_beat_label <= 30)]
y_beats = librosa.clicks(times=gt_beat_label_slice, sr=sr, click_freq=1000, length=len(y_slice))
ipd.display(ipd.Audio(y_slice + y_beats, rate=sr))





# Beat Tracking By Dynamic Programing


Let's do beat tracking on the intro part which does not have vocal sources.

In [ ]:
# get the audio file
audio_url = "https://www.audiolabs-erlangen.de/resources/MIR/FMP/data/C5/FMP_C5_Audio_Beatles_LetItBe_Beatles_1970-LetItBe-06.wav"
response = requests.get(audio_url)
audio_bytes = io.BytesIO(response.content)

# take the intro part (piano only)
start_sec = 0
duration = 11
y_slice, _ = librosa.load(audio_bytes, sr=sr, offset=start_sec, duration=duration)
gt_beats_slice = gt_beat_label[(gt_beat_label >= start_sec) & (gt_beat_label <= start_sec + duration)] - start_sec

# run the beat tracking
onset_env = librosa.onset.onset_strength(y=y_slice, sr=sr)
tempo_pred, beat_frames_pred = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr)
pred_beats_slice = librosa.frames_to_time(beat_frames_pred, sr=sr)

# print tempo
print('predicted tempo (BPM)', tempo_pred)

# plot the ground truth beats and predicted beats
plt.figure(figsize=(15, 2))
librosa.display.waveshow(y_slice, sr=sr, alpha=0.5, color='gray')
plt.vlines(gt_beats_slice, -1, 1, color='g', linestyle='-', linewidth=2, label='Ground Truth (Reference)')
plt.vlines(pred_beats_slice, -1, 1, color='r', linestyle='--', linewidth=2, label='Predicted (Baseline)')
plt.title(f'Beat Tracking Comparison: {start_sec}s - {start_sec + duration}s')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# report the result using "mir_eval"
scores = mir_eval.beat.evaluate(gt_beats_slice, pred_beats_slice)

print("="*40)
print(f"{'MIR Evaluation Metric':<25} | {'Score':<10}")
print("-"*40)
print(f"{'F-measure':<25} | {scores['F-measure']:.4f}")
print(f"{'CMLc (Correct Cont.)':<25} | {scores['Correct Metric Level Continuous']:.4f}")
print(f"{'CMLt (Correct Total)':<25} | {scores['Correct Metric Level Total']:.4f}")
print(f"{'AMLc (Any Metric Cont.)':<25} | {scores['Any Metric Level Continuous']:.4f}")
print(f"{'AMLt (Any Metric Total)':<25} | {scores['Any Metric Level Total']:.4f}")
print("="*40)


# sonify the estimated beat
y_beats = librosa.clicks(times=pred_beats_slice, sr=sr, click_freq=1000, length=len(y_slice))
ipd.display(ipd.Audio(y_slice + y_beats, rate=sr))


The simple implementation works perfectly on the intro part that has only piano.

Let's take the vocal part and use the same beat tracking code.

In [ ]:
# get the audio file
audio_url = "https://www.audiolabs-erlangen.de/resources/MIR/FMP/data/C5/FMP_C5_Audio_Beatles_LetItBe_Beatles_1970-LetItBe-06.wav"
response = requests.get(audio_url)
audio_bytes = io.BytesIO(response.content)

# take the verse part (vocal + piano)
start_sec = 15
duration = 11
y_slice, _ = librosa.load(audio_bytes, sr=sr, offset=start_sec, duration=duration)
gt_beats_slice = gt_beat_label[(gt_beat_label >= start_sec) & (gt_beat_label <= start_sec + duration)] - start_sec

# run the beat tracking
onset_env = librosa.onset.onset_strength(y=y_slice, sr=sr)
tempo_pred, beat_frames_pred = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr)
pred_beats_slice = librosa.frames_to_time(beat_frames_pred, sr=sr)

# print tempo
print('predicted tempo (BPM)', tempo_pred)

# plot the ground truth beats and predicted beats
plt.figure(figsize=(15, 2))
librosa.display.waveshow(y_slice, sr=sr, alpha=0.5, color='gray')
plt.vlines(gt_beats_slice, -1, 1, color='g', linestyle='-', linewidth=2, label='Ground Truth (Reference)')
plt.vlines(pred_beats_slice, -1, 1, color='r', linestyle='--', linewidth=2, label='Predicted (Baseline)')
plt.title(f'Beat Tracking Comparison: {start_sec}s - {start_sec + duration}s')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# report the result using "mir_eval"
scores = mir_eval.beat.evaluate(gt_beats_slice, pred_beats_slice)

print("="*40)
print(f"{'MIR Evaluation Metric':<25} | {'Score':<10}")
print("-"*40)
print(f"{'F-measure':<25} | {scores['F-measure']:.4f}")
print(f"{'CMLc (Correct Cont.)':<25} | {scores['Correct Metric Level Continuous']:.4f}")
print(f"{'CMLt (Correct Total)':<25} | {scores['Correct Metric Level Total']:.4f}")
print(f"{'AMLc (Any Metric Cont.)':<25} | {scores['Any Metric Level Continuous']:.4f}")
print(f"{'AMLt (Any Metric Total)':<25} | {scores['Any Metric Level Total']:.4f}")
print("="*40)


# sonify the estimated beat
y_beats = librosa.clicks(times=pred_beats_slice, sr=sr, click_freq=1000, length=len(y_slice))
ipd.display(ipd.Audio(y_slice + y_beats, rate=sr))


The accuracy is aweful! Why the tmepo increases a lot? How can you improve the performance?

# Chord Recognition by HMM

Let's do chord recognition with the template-basd approach.

In [ ]:
import librosa, libfmp.b, libfmp.c5, libfmp.c4, mir_eval
import numpy as np, pandas as pd, requests, io, matplotlib.pyplot as plt

audio_url = "https://www.audiolabs-erlangen.de/resources/MIR/FMP/data/C5/FMP_C5_Audio_Beatles_LetItBe_Beatles_1970-LetItBe-06.wav"
chord_url = "https://www.audiolabs-erlangen.de/resources/MIR/FMP/data/C5/FMP_C5_Audio_Beatles_LetItBe_Beatles_1970-LetItBe-06_Chords_simplified.csv"

start_sec = 15
duration = 11
end_sec = start_sec+duration
sr = 22050
y, sr = librosa.load(io.BytesIO(requests.get(audio_url).content), sr=sr, offset=start_sec, duration=duration)

# take the chord labels
ann_all, _ = libfmp.c4.read_structure_annotation(io.BytesIO(requests.get(chord_url).content))
ann = []
for s in ann_all:
    if s[1] > start_sec and s[0] < end_sec:
        new_start = max(0, s[0] - start_sec)
        new_end = min(duration, s[1] - start_sec)
        ann.append([new_start, new_end, s[2]])

# extract the chroma feature
hop_length = 512
chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)

# caculate time intervals for frame-level prediction
frames = np.arange(chroma.shape[1])
frame_times = librosa.frames_to_time(frames, sr=sr, hop_length=hop_length)
frame_intervals = np.vstack([frame_times, np.append(frame_times[1:], duration)]).T

## template-based approach
chord_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B',
                'C:min', 'C#:min', 'D:min', 'D#:min', 'E:min', 'F:min', 'F#:min', 'G:min', 'G#:min', 'A:min', 'A#:min', 'B:min']
chord_sim, chord_binary_temp = libfmp.c5.chord_recognition_template(chroma, norm_sim='1')

# obtain chord labels as text
chord_id_temp = np.argmax(chord_binary_temp, axis=0)
est_labels_temp = [chord_labels[i] for i in chord_id_temp]

# merge frame-level time intervals to segment-level time intervals based on chord predictions
est_temp_seg = mir_eval.chord.merge_chord_intervals(frame_intervals, est_labels_temp)

# take segment-level chord predictions
from itertools import groupby
est_labels_temp = [key for key, group in groupby(est_labels_temp)]

# evluation
def calculate_metrics(ref_int, ref_lab, est_int, est_lab):
    scores = mir_eval.chord.evaluate(ref_int, ref_lab, est_int, est_lab)
    score_csr = scores['majmin']
    score_seg = scores["seg"]
    return score_csr, score_seg

ref_intervals = np.array([[a[0], a[1]] for a in ann])
ref_labels = [a[2] for a in ann]

score_csr_temp, score_seg_temp = calculate_metrics(ref_intervals, ref_labels, est_temp_seg, est_labels_temp)

print(f"--- Evaluation Results ---")
print(f"[Template] CSR: {score_csr_temp:.4f} | Segmentation_Accuracy: {score_seg_temp:.4f}")

### Visualize the chord predictions ###
fig, ax = plt.subplots(4, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [1.5, 2.5, 1, 1]})

# (1) Chromagram (Input)
img0 = librosa.display.specshow(chroma, x_axis='time', y_axis='chroma', ax=ax[0], hop_length=hop_length)
ax[0].set_title('1. Chromagram (Input Feature)')
fig.colorbar(img0, ax=ax[0])

# (2) Chord Similarity Matrix (Emission)
img1 = librosa.display.specshow(chord_sim, x_axis='time', ax=ax[1], hop_length=hop_length)
ax[1].set_yticks(np.arange(len(chord_labels)))
ax[1].set_yticklabels(chord_labels, fontsize=7)
ax[1].set_title('2. Chord Similarity Matrix (Observation)')
fig.colorbar(img1, ax=ax[1])

# (3) Ground Truth
libfmp.b.plot_segments(ann, ax=ax[2])
ax[2].set_title('3. Ground Truth (Reference)')

# make the chord prediction format the same as that of the ground truth
est_temp_seg2 = [[i[0], i[1], l] for i, l in zip(est_temp_seg, est_labels_temp)]

# (4) Template Result
libfmp.b.plot_segments(est_temp_seg2, ax=ax[3])
ax[3].set_title(f'4. Template-based Prediction')

plt.xlim([0, duration])
plt.tight_layout()
plt.show()



The accuracy is not very low but there are many sudden jumps between chord predictions (the segmentation accuracy is low).

So, let's implement the HMM-based approach to have more smooth predictions and higher accuracy.